**Before you start:** run `uv sync` in the project folder, then open this notebook with `uv run --with jupyter jupyter lab`, so it uses the project's packages from `pyproject.toml` and `uv.lock`.

### Step 1: Check the environment
This cell loads the Atlantis game and prints what the agent can do and see:
- **Action space:** `Discrete(4)` means 4 possible actions (do nothing, fire, move right and fire, move left and fire).
- **Frame shape:** `(210, 160, 3)` is the raw game picture: 210 pixels high, 160 wide, in colour (3 channels).

This raw picture is too big for training, so the next cell shrinks it.

In [ ]:
import gymnasium as gym
import ale_py

# Register the Atari environments with Gymnasium
gym.register_envs(ale_py)

env = gym.make("ALE/Atlantis-v5")

print("Action space:", env.action_space)
print("Observation space:", env.observation_space.shape)

obs, info = env.reset()
print("First frame shape:", obs.shape)

env.close()

Action space: Discrete(4)
Observation space: (210, 160, 3)
First frame shape: (210, 160, 3)


### Step 2: Prepare the environment (`make_env`)
`make_env()` builds the environment the agent will actually train on. It does three things:
1. **Frame skip (4):** each action is repeated for 4 frames, so the agent decides less often and learns faster.
2. **Grayscale and 84x84:** colour is not needed to play, and a smaller picture is much faster for the network.
3. **Stack 4 frames:** one frame is a still picture, but 4 frames in a row show which way things are moving.

The result is an observation of shape `(4, 84, 84)`. `frameskip=1` in `gym.make` turns off the game's own skipping so it is not done twice.

In [ ]:
import gymnasium as gym
import ale_py
import numpy as np
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation

gym.register_envs(ale_py)

def make_env(render_mode=None):
    # frameskip=1 here, because AtariPreprocessing does the frame skipping itself
    env = gym.make("ALE/Atlantis-v5", frameskip=1, render_mode=render_mode)
    # grayscale + resize to 84x84 + skip 4 frames per action
    env = AtariPreprocessing(env, frame_skip=4, screen_size=84, grayscale_obs=True)
    # stack the last 4 frames so the agent can see motion
    env = FrameStackObservation(env, stack_size=4)
    return env

# Test it
env = make_env()
obs, info = env.reset()
print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Number of actions:", env.action_space.n)

obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("After one step:", obs.shape, "reward:", reward)
env.close()

Observation shape: (4, 84, 84)
Observation dtype: uint8
Number of actions: 4
After one step: (4, 84, 84) reward: 0.0


### Step 3: The Q-network (`QNetwork`)
This is the agent's "brain". It takes the 4 stacked frames and outputs **one number per action** (the Q-value). A higher number means the agent expects more score from that action.
- **3 convolutional layers:** find patterns in the picture, such as enemy ships.
- **2 fully connected layers:** turn those patterns into the 4 Q-values.
- **`x / 255.0`:** scales pixel values from 0-255 down to 0-1, which makes training steadier.

The network starts with random weights (no pretrained model). The test at the end feeds in a fake batch of 32 frames and checks that 32 rows of 4 values come out.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class QNetwork(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        x = x.float() / 255.0  # scale pixels from 0-255 to 0-1
        return self.fc(self.conv(x))


# Test: a fake batch of 32 observations should give 32 rows of 4 Q-values
net = QNetwork(4).to(device)
test_input = torch.zeros(32, 4, 84, 84, dtype=torch.uint8).to(device)
print("Q-values shape:", net(test_input).shape)

Using device: cuda
Q-values shape: torch.Size([32, 4])


### Step 4: The replay buffer (`ReplayBuffer`)
This is the agent's **memory**. After every step it stores one experience: the state, the action taken, the reward, the next state, and whether the game ended.
- **`add`:** saves a new experience. When the memory is full, the oldest one is overwritten.
- **`sample`:** returns a random batch of past experiences for training.

Random batches matter because consecutive frames look almost the same, and learning from them in order would make training unstable. Frames are stored as small `uint8` numbers to save memory.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.index = 0   # where the next experience goes
        self.size = 0    # how many experiences are stored
        # frames stay uint8 to save memory
        self.states = np.zeros((capacity, 4, 84, 84), dtype=np.uint8)
        self.next_states = np.zeros((capacity, 4, 84, 84), dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)

    def add(self, state, action, reward, next_state, done):
        self.states[self.index] = state
        self.actions[self.index] = action
        self.rewards[self.index] = reward
        self.next_states[self.index] = next_state
        self.dones[self.index] = done
        self.index = (self.index + 1) % self.capacity  # overwrite oldest when full
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.from_numpy(self.states[idx]).to(device),
            torch.from_numpy(self.actions[idx]).to(device),
            torch.from_numpy(self.rewards[idx]).to(device),
            torch.from_numpy(self.next_states[idx]).to(device),
            torch.from_numpy(self.dones[idx]).to(device),
        )

    def __len__(self):
        return self.size

### Step 5: Random agent baseline
This plays 50 full games where every action is chosen **at random**, with no learning. The average score shows what "no learning" looks like, so a trained agent can be compared against it. (`max_steps=10_000` stops any game from running forever, the same limit as in training.)

The games are seeded, so the result is the same every time the cell runs: the random button presses use seed 42, and game *n* starts from seed 42 + *n*, the same game seeds `play.py` uses. This average is the random-play score in the README, the experiment tables and the red line in the training plots.

In [ ]:
SEED = 42  # the team seed, also used by train.py and play.py


def run_random_agent(episodes=50, max_steps=10_000, seed=SEED):
    env = make_env()
    env.action_space.seed(seed)  # fixes the random button presses
    scores = []
    for ep in range(episodes):
        # fixes how each game starts: game n uses seed + n, like play.py
        env.reset(seed=seed + ep + 1)
        total_reward, steps, done = 0, 0, False
        while not done and steps < max_steps:
            _, reward, terminated, truncated, _ = env.step(env.action_space.sample())
            total_reward += reward
            steps += 1
            done = terminated or truncated
        scores.append(total_reward)
        print(f"Episode {ep + 1}: score = {total_reward:,.0f}, steps = {steps:,}")
    env.close()
    print(f"\nRandom agent over {episodes} games (seed {seed}):")
    print(f"  average score = {np.mean(scores):,.0f}")
    print(f"  lowest = {np.min(scores):,.0f}, highest = {np.max(scores):,.0f}")
    return scores


random_scores = run_random_agent()

Episode 1: score = 13,700, steps = 972
Episode 2: score = 28,700, steps = 1,656
Episode 3: score = 26,400, steps = 1,828
Episode 4: score = 11,700, steps = 943
Episode 5: score = 33,600, steps = 2,058
Episode 6: score = 11,400, steps = 1,061
Episode 7: score = 20,900, steps = 1,262
Episode 8: score = 15,900, steps = 1,078
Episode 9: score = 19,800, steps = 1,718
Episode 10: score = 9,800, steps = 975
Episode 11: score = 8,900, steps = 872
Episode 12: score = 5,200, steps = 608
Episode 13: score = 24,400, steps = 1,711
Episode 14: score = 15,200, steps = 1,146
Episode 15: score = 19,700, steps = 1,525
Episode 16: score = 12,600, steps = 1,182
Episode 17: score = 12,400, steps = 1,385
Episode 18: score = 9,100, steps = 832
Episode 19: score = 24,900, steps = 1,782
Episode 20: score = 14,500, steps = 998
Episode 21: score = 19,100, steps = 1,401
Episode 22: score = 10,500, steps = 1,080
Episode 23: score = 11,500, steps = 944
Episode 24: score = 21,900, steps = 1,730
Episode 25: score = 1